In [1]:
import numpy as np
import pandas as pd
import re
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input,LSTM,Embedding,Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import contractions

In [2]:
df = pd.read_csv("Dataset_English_Hindi.csv")

In [3]:
df.drop_duplicates(inplace=True)

In [4]:
#lower case
df['English'] = df['English'].str.lower()

In [5]:
#remove non english words
def remove_non_english(text):
    pattern = r'[^a-zA-Z0-9\s]'
    text = re.sub(pattern, '', text)
    return text

df['English'] = df['English'].astype(str)
df['English'] = df['English'].apply(remove_non_english)

In [6]:
#remove non hindi words
def remove_non_hindi(text):
    pattern = r'[^\u0900-\u097F\s]'
    text = re.sub(pattern, '', text)
    return text

df['Hindi'] = df['Hindi'].astype(str)
df['Hindi'] = df['Hindi'].apply(remove_non_hindi)

In [7]:
#remove urls
def remove_url(text):
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub(r'',text)

df['English'] = df['English'].apply(remove_url)
df['Hindi'] = df['Hindi'].apply(remove_url)

In [8]:
#handle chat words
chat_words = {
    "AFAIK": "As Far As I Know",
    "AFK": "Away From Keyboard",
    "ASAP": "As Soon As Possible",
    "ATK": "At The Keyboard",
    "ATM": "At The Moment",
    "A3": "Anytime, Anywhere, Anyplace",
    "BAK": "Back At Keyboard",
    "BBL": "Be Back Later",
    "BBS": "Be Back Soon",
    "BFN": "Bye For Now",
    "B4N": "Bye For Now",
    "BRB": "Be Right Back",
    "BRT": "Be Right There",
    "BTW": "By The Way",
    "B4": "Before",
    "CU": "See You",
    "CUL8R": "See You Later",
    "CYA": "See You",
    "FAQ": "Frequently Asked Questions",
    "FC": "Fingers Crossed",
    "FWIW": "For What It's Worth",
    "FYI": "For Your Information",
    "GAL": "Get A Life",
    "GG": "Good Game",
    "GN": "Good Night",
    "GMTA": "Great Minds Think Alike",
    "GR8": "Great!",
    "G9": "Genius",
    "IC": "I See",
    "ICQ": "I Seek you (also a chat program)",
    "ILU": "I Love You",
    "IMHO": "In My Honest/Humble Opinion",
    "IMO": "In My Opinion",
    "IOW": "In Other Words",
    "IRL": "In Real Life",
    "KISS": "Keep It Simple, Stupid",
    "LDR": "Long Distance Relationship",
    "LMAO": "Laugh My A.. Off",
    "LOL": "Laughing Out Loud",
    "LTNS": "Long Time No See",
    "L8R": "Later",
    "MTE": "My Thoughts Exactly",
    "M8": "Mate",
    "NRN": "No Reply Necessary",
    "OIC": "Oh I See",
    "PITA": "Pain In The A..",
    "PRT": "Party",
    "PRW": "Parents Are Watching",
    "QPSA": "Que Pasa?",
    "ROFL": "Rolling On The Floor Laughing",
    "ROFLOL": "Rolling On The Floor Laughing Out Loud",
    "ROTFLMAO": "Rolling On The Floor Laughing My A.. Off",
    "SK8": "Skate",
    "STATS": "Your sex and age",
    "ASL": "Age, Sex, Location",
    "THX": "Thank You",
    "TTFN": "Ta-Ta For Now!",
    "TTYL": "Talk To You Later",
    "U": "You",
    "U2": "You Too",
    "U4E": "Yours For Ever",
    "WB": "Welcome Back",
    "WTF": "What The F...",
    "WTG": "Way To Go!",
    "WUF": "Where Are You From?",
    "W8": "Wait...",
    "7K": "Sick:-D Laughter",
    "TFW": "That feeling when",
    "MFW": "My face when",
    "MRW": "My reaction when",
    "IFYP": "I feel your pain",
    "LOL": "Laughing out loud",
    "TNTL": "Trying not to laugh",
    "JK": "Just kidding",
    "IDC": "I don’t care",
    "ILY": "I love you",
    "IMU": "I miss you",
    "ADIH": "Another day in hell",
    "IDC": "I don’t care",
    "ZZZ": "Sleeping, bored, tired",
    "WYWH": "Wish you were here",
    "TIME": "Tears in my eyes",
    "BAE": "Before anyone else",
    "FIMH": "Forever in my heart",
    "BSAAW": "Big smile and a wink",
    "BWL": "Bursting with laughter",
    "LMAO": "Laughing my a** off",
    "BFF": "Best friends forever",
    "CSL": "Can’t stop laughing",
}

In [9]:
def chat_conversion(text):
    new_text = []
    for w in text.split():
        if w.upper() in chat_words.keys():
            new_text.append(chat_words[w.upper()].lower())
        else:
            new_text.append(w)
    return ' '.join(new_text)

df['English'] = df['English'].apply(chat_conversion)

In [10]:
#remove emojis
def remove_emoji(text):
    emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

df['English'] = df['English'].apply(remove_emoji)
df['Hindi'] = df['Hindi'].apply(remove_emoji)

In [11]:
#expand contractions
def expand_contractions(text):
    expanded_text = contractions.fix(text)
    return expanded_text

df['English'] = df['English'].apply(expand_contractions)

In [12]:
#tokenizer english
tok_eng = Tokenizer()
tok_eng.fit_on_texts(df['English'])
x = tok_eng.texts_to_sequences(df['English'])
x = pad_sequences(x)

In [13]:
#tokenizer hindi
tok_hin = Tokenizer()
tok_hin.fit_on_texts(df['Hindi'])
y = tok_hin.texts_to_sequences(df['Hindi'])
y = pad_sequences(y)

In [14]:
vocab_eng = len(tok_eng.word_index) + 1
vocab_hin = len(tok_hin.word_index) + 1

In [15]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [16]:
y_train_in = y_train[:, :-1]
y_train_out = y_train[:, 1:]

In [17]:
#encoder
enc_input = Input(shape=(x_train.shape[1],))
enc_embed = Embedding(input_dim=vocab_eng, output_dim=256)(enc_input)
_,h,c = LSTM(256, return_state=True)(enc_embed)

In [18]:
#decoder
dec_input = Input(shape=(y_train.shape[1],))
dec_embed = Embedding(input_dim=vocab_hin, output_dim=256)(dec_input)
dec_lstm = LSTM(256, return_sequences=True)(dec_embed, initial_state=[h,c])
dec_out = Dense(vocab_hin, activation='softmax')(dec_lstm)

In [19]:
#model
model = Model([enc_input, dec_input], dec_out)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 401)               │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_1 (InputLayer)    │ (None, 416)               │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding (Embedding)         │ (None, 401, 256)          │      19,012,864 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding_1 (Embedding)       │ (None, 416, 256)          │      19,715,072 │ input_layer_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm (LSTM)                   │ [(None, 256), (None,      │         525,312 │ embedding[0][0]            │
│                               │ 256), (None, 256)]        │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_1 (LSTM)                 │ (None, 416, 256)          │         525,312 │ embedding_1[0][0],         │
│                               │                           │                 │ lstm[0][1], lstm[0][2]     │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 416, 77012)        │      19,792,084 │ lstm_1[0][0]               │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 59,570,644 (227.24 MB)

 Trainable params: 59,570,644 (227.24 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
model_checkpoint = ModelCheckpoint('model_checkpoint.keras', save_best_only=True)

history = model.fit(
    x=[x_train, y_train],
    y=y_train,
    batch_size=16,
    epochs=2,
    validation_data = ([x_test, y_test], y_test),
    callbacks = [early_stopping, model_checkpoint]
)

Epoch 1/2
   5/6385 ━━━━━━━━━━━━━━━━━━━━ 20:15:23 11s/step - accuracy: 0.5224 - loss: 11.1201

KeyboardInterrupt: 

In [21]:
model.save("/content/drive/MyDrive/eng_hin_translator/my_model.keras")
model.save_weights("/content/drive/MyDrive/eng_hin_translator/my_model.weights.h5")
with open("/content/drive/MyDrive/eng_hin_translator/my_model_architecture.json", 'w') as f:
    f.write(model.to_json())

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/eng_hin_translator/my_model.keras'

In [22]:
import subprocess
import sys

def check_gpu():
    # Check via nvidia-smi (for NVIDIA GPUs)
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
             "--format=csv,noheader"],
            capture_output=True, text=True, check=True
        )
        gpus = result.stdout.strip().split("\n")
        print(f"Found {len(gpus)} NVIDIA GPU(s):")
        for i, gpu in enumerate(gpus):
            print(f"  GPU {i}: {gpu}")
    except (subprocess.CalledProcessError, FileNotFoundError):
        print("No NVIDIA GPU detected or nvidia-smi not installed.")

check_gpu()

No NVIDIA GPU detected or nvidia-smi not installed.


In [23]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [25]:
import cupy as cp

try:
    n_gpus = cp.cuda.runtime.getDeviceCount()
    print(f"Number of CUDA GPUs: {n_gpus}")
    for i in range(n_gpus):
        props = cp.cuda.runtime.getDeviceProperties(i)
        print(f"  GPU {i}: {props['name'].decode()}")
except cp.cuda.runtime.CUDARuntimeError:
    print("No CUDA GPU available.")


No CUDA GPU available.


In [26]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU is available. Found {len(gpus)} GPU(s):")
    for gpu in gpus:
        print(f"  - {gpu.name}")
else:
    print("No GPU available. Using CPU.")

No GPU available. Using CPU.


In [27]:
import torch

if torch.cuda.is_available():
    print(f"GPU is available: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current GPU: {torch.cuda.current_device()}")
else:
    print("No GPU available. Using CPU.")

No GPU available. Using CPU.


In [28]:
import torch

if torch.xpu.is_available():
    print(f"Intel GPU is available!")
    print(f"Number of Intel GPUs: {torch.xpu.device_count()}")
    print(f"Current device: {torch.xpu.current_device()}")
else:
    print("No Intel GPU detected. Check driver installation.")

No Intel GPU detected. Check driver installation.


In [29]:
import subprocess
import platform

def check_intel_gpu():
    os_name = platform.system()
    
    if os_name == "Windows":
        result = subprocess.run(
            ["wmic", "path", "win32_VideoController", "get", "name"],
            capture_output=True, text=True
        )
        output = result.stdout
    else:  # Linux
        result = subprocess.run(
            ["lspci"], capture_output=True, text=True
        )
        output = result.stdout

    intel_gpus = [line for line in output.splitlines() if "Intel" in line and "Graphics" in line]
    
    if intel_gpus:
        print(f"Intel GPU(s) found:")
        for gpu in intel_gpus:
            print(f"  - {gpu.strip()}")
    else:
        print("No Intel GPU detected.")

check_intel_gpu()


Intel GPU(s) found:
  - Intel(R) UHD Graphics 770


In [30]:
import dpctl

# List all available SYCL devices including Intel GPUs
for device in dpctl.get_devices():
    print(device)

# Specifically check for GPU
gpu_devices = dpctl.get_devices(device_type="gpu")
if gpu_devices:
    print(f"Intel GPU found: {gpu_devices[0].name}")


ModuleNotFoundError: No module named 'dpctl'

In [31]:
torch.xpu.is_available()

False

In [32]:
import tensorflow as tf
import intel_extension_for_tensorflow as itex

ModuleNotFoundError: No module named 'intel_extension_for_tensorflow'